### AFRE Performance Analysis

This notebook loads AFRE and baseline CSV logs, computes aggregates, and visualizes trade-offs:
- FPS vs. Resolution
- Accuracy (confidence proxy) vs. Precision
- Throughput vs. GPU load
It also builds a summary table and estimates throughput uplift and accuracy delta.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from adaptive_frame_rate_engine.metrics import read_csv_rows, aggregate_metrics, compare_accuracy_vs_baseline

logs_dir = Path('../logs')
run_prefix = None  # set to a specific run prefix if needed

# Load latest AFRE and baseline logs
csvs = sorted(list(logs_dir.glob('*.csv')))
afres = [c for c in csvs if c.name.endswith('_afre.csv')]
bases = [c for c in csvs if c.name.endswith('_baseline.csv')]

afre_csv = afres[-1] if afres else None
base_csv = bases[-1] if bases else None
print('AFRE CSV:', afre_csv)
print('BASE CSV:', base_csv)

af_rows = read_csv_rows(afre_csv) if afre_csv else []
bl_rows = read_csv_rows(base_csv) if base_csv else []

# Create DataFrames
af_df = pd.DataFrame(af_rows)
bl_df = pd.DataFrame(bl_rows)

for df in [af_df, bl_df]:
	for col in ['fps','latency_ms','gpu_util','gpu_mem_mb','resolution','confidence']:
		if col in df.columns:
			df[col] = pd.to_numeric(df[col], errors='coerce')

# Aggregates
agg_af = aggregate_metrics(af_rows) if af_rows else {}
agg_bl = aggregate_metrics(bl_rows) if bl_rows else {}
acc_delta = compare_accuracy_vs_baseline(af_rows, bl_rows)
print('AFRE aggregate:', agg_af)
print('Baseline aggregate:', agg_bl)
print('Accuracy delta (AFRE - Baseline) %:', acc_delta)

# Plots
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
if not af_df.empty:
	af_df.groupby('resolution')['fps'].mean().plot(kind='bar', title='FPS vs Resolution (AFRE)')
	plt.ylabel('FPS')
else:
	plt.title('No AFRE data')

plt.subplot(1,3,2)
if not af_df.empty and 'precision' in af_df.columns:
	af_df.groupby('precision')['confidence'].mean().plot(kind='bar', title='Confidence vs Precision (AFRE)')
	plt.ylabel('Confidence (proxy)')
else:
	plt.title('No AFRE data')

plt.subplot(1,3,3)
if not af_df.empty:
	plt.scatter(af_df['gpu_util'], af_df['fps'], s=8)
	plt.title('Throughput vs GPU Load (AFRE)')
	plt.xlabel('GPU Util (%)')
	plt.ylabel('FPS')
else:
	plt.title('No AFRE data')

plt.tight_layout()
plt.show()

# Summary table
summary_rows = []
if agg_af:
	summary_rows.append({'mode':'AFRE', **agg_af})
if agg_bl:
	summary_rows.append({'mode':'Baseline', **agg_bl})
summary_df = pd.DataFrame(summary_rows)
summary_df['acc_delta_vs_baseline_%'] = acc_delta
summary_df
